In [1]:
from datetime import date
import pandas as pd
import pro_sports_transactions as pst
import asyncio
import nest_asyncio; nest_asyncio.apply() # needed for running code in jupyter


In [19]:
async def search_transactions(starting_row, transaction_type) -> str:
    return await pst.Search(
        league = pst.League.NBA,
        transaction_types = [transaction_type], # Needs to list, hence []
        start_date = date.fromisoformat("2023-08-01"), # Optional
        end_date = date.today(),
        starting_row = starting_row
    ).get_dict()

In [43]:
# df = asyncio.run(search_transactions()) # use this code when not running in jupyter
df = pd.DataFrame()

pst_page = asyncio.get_event_loop()
for t_t in pst.TransactionType:
    pages = pst_page.run_until_complete(search_transactions(0, t_t))['pages']
    
    for page in range(pages):
        df_t = pst_page.run_until_complete(search_transactions(page * 25, t_t))
        df_t = pd.DataFrame(df_t['transactions'])
        df_t['transaction_type'] = t_t.name
        df = pd.concat([df, df_t], axis=0, ignore_index=True)

df['acc_req'] = ['Acquired' if len(row[1]['Relinquished'])==0 else 'Relinquished' for row in df.iterrows()]
df['player'] = [row[1]['Acquired'] if len(row[1]['Relinquished'])==0 else row[1]['Relinquished'] for row in df.iterrows()]
df['player'] = df['player'].str.removeprefix('• ')
df.columns = df.columns.str.lower()
df = df[['date', 'transaction_type', 'team', 'player', 'acc_req', 'notes']]

In [44]:
df.to_csv('pst_init.csv', index=False, na_rep=None)

In [45]:
df

,date,transaction_type,team,player,acc_req,notes,acquired,relinquished
0,2023-08-02,Disciplinary,Spurs,Devonte' Graham,Relinquished,suspended by NBA for 2 games for pleading guil...,,• Devonte' Graham
1,2023-08-09,Disciplinary,Timberwolves,Anthony Edwards,Relinquished,"fined $50,000 by NBA for recklessly swinging a...",,• Anthony Edwards
2,2023-08-22,Disciplinary,76ers,James Harden,Relinquished,fined $100K by NBA for comments on August 14 a...,,• James Harden
3,2023-08-02,Injury,76ers,Montrezl Harrell,Relinquished,torn ACL in knee (out indefinitely),,• Montrezl Harrell
4,2023-08-05,Injury,Nuggets,Vlatko Cancar,Relinquished,torn ACL in knee (out indefinitely),,• Vlatko Cancar
...,...,...,...,...,...,...,...,...
61,2023-08-31,Movement,Grizzlies,Gregory Jackson II,Acquired,signed second round pick to a two way contract,• Gregory Jackson II,
62,2023-08-31,Movement,Grizzlies,Shaquille Harrison,Acquired,signed free agent,• Shaquille Harrison,
63,2023-08-31,Movement,Kings,JaVale McGee,Acquired,signed free agent to a 1-year contract,• JaVale McGee,
64,2023-09-01,Movement,Bucks,Alex Antetokounmpo,Acquired,signed free agent,• Alex Antetokounmpo,
